<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "324"
iso_country = "Guinea"
year = 2016
import_portfolio = "h4_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 8.600*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Guinea': 

Number of trade records present in dataset: 15882
Number of partner countries present in dataset: 139
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2109.7m
Total weight within the dataset: 1948.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,324,Guinea,156,China,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,18611.91564,NaN,18611.91564,kg,21958.922,21958.922,3383.646263,21101.0,13111.549471,44364.845374,['Container'],['Container'],154.412552,153.983794,154.024104,154.024104,154.250756,154.250756,162.301715,299.383799,177.233834,165.724709,166.316115,170.353415,158.612288,289.754174,187.249303,171.855187,167.478787,176.030595,155.372172,319.920787,232.330649,204.102092,188.041968,183.913327,152.191238,337.302246,274.575485,253.533954,204.756771,197.854956,149.073411,353.562726,336.643301,342.358298,215.492799,208.192259,145.894927,341.887872,330.654295,353.060862,275.217321,212.174336,142.716382,300.194876,308.887798,333.418786,245.165891,214.720008,139.671729,270.531811,276.906701,323.064784,264.423627,216.378311,2711
1,1,324,Guinea,702,Singapore,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,104.60879,NaN,104.60879,kg,239.758,239.758,19.017878,16917.0,10511.733207,199.910860,['Container'],['Container'],0.695793,0.693861,0.694043,0.694043,0.695064,0.695064,0.731342,1.349043,0.798627,0.746766,0.749431,0.767623,0.714717,1.305651,0.843757,0.774391,0.754670,0.793205,0.700117,1.441584,1.046897,0.919697,0.847329,0.828725,0.685784,1.519906,1.237255,1.142440,0.922647,0.891547,0.671734,1.593176,1.516936,1.542689,0.971024,0.938128,0.657412,1.540569,1.489950,1.590915,1.240147,0.956071,0.643089,1.352698,1.391868,1.502407,1.104733,0.967542,0.629370,1.219034,1.247759,1.455751,1.191510,0.975015,2711


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $509.3m (24.1%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               208045733.3  418561228.7      9.9     21.5
08 | Fruit and nuts, edib  155531878.1  102074105.2      7.4      5.2
17 | Sugars and sugar con   42301222.0   87867138.4      2.0      4.5
22 | Beverages, spirits a   19505293.1   31289609.6      0.9      1.6
02 | Meat and edible meat   18038805.3    9794913.8      0.9      0.5
15 | Animal, vegetable or   17707746.4   19820575.7      0.8      1.0
19 | Preparations of cere   11271220.8    4013852.8      0.5      0.2
21 | Miscellaneous edible    9336731.0    3339831.3      0.4      0.2
04 | Dairy produce; birds    6805914.3    3145097.6      0.3      0.2
11 | Products of the mill    4812968.6    9138386.9      0.2      0.5
20 | Preparations of vege    4475709.5    4004752.4      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $488.5m (23.2%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       387363114.3  460423804.9     18.4     23.6
Food & Beverages  100836564.7  133392168.8      4.8      6.8
Fishing              280786.7     129838.3      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $457.2m (21.7%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
V_F - Vegetables, fruit, nuts  159045844.3  108334233.5      7.5      5.6
PCR - Processed rice           152472588.5  264844886.2      7.2     13.6
SGR - Sugar                     39436339.5   86525114.3      1.9      4.4
OFD - Food products n.e.c.      35651814.8   20091494.3      1.7      1.0
PDR - Paddy rice                25377592.7   44080878.5      1.2      2.3
VOL - Vegetable oils and fats   18297738.1   21198789.0      0.9      1.1
OMT - Meat products n.e.c.      16009756.3    8541518.1      0.8      0.4
MIL - Dairy products             6836131.3    3141078.7      0.3      0.2
CMT - Bovine meat prods          2688265.9    1384493.2      0.1      0.1
OCR - Crops n.e.c.               1230487.6     325336.8      0.1      0.0
GRO - Cereal grains n.e.c.        119112.2    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 24.1% 
GTAP 21.7% 
EORA 23.2%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46378/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,324,Guinea,156,China,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,18611.91564,NaN,18611.91564,kg,21958.922,21958.922,3383.646263,21101.0,13111.549471,44364.845374,['Container'],['Container'],154.412552,153.983794,154.024104,154.024104,154.250756,154.250756,162.301715,299.383799,177.233834,165.724709,166.316115,170.353415,158.612288,289.754174,187.249303,171.855187,167.478787,176.030595,155.372172,319.920787,232.330649,204.102092,188.041968,183.913327,152.191238,337.302246,274.575485,253.533954,204.756771,197.854956,149.073411,353.562726,336.643301,342.358298,215.492799,208.192259,145.894927,341.887872,330.654295,353.060862,275.217321,212.174336,142.716382,300.194876,308.887798,333.418786,245.165891,214.720008,139.671729,270.531811,276.906701,323.064784,264.423627,216.378311,2711,NaN
1,1,324,Guinea,702,Singapore,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,104.60879,NaN,104.60879,kg,239.758,239.758,19.017878,16917.0,10511.733207,199.910860,['Container'],['Container'],0.695793,0.693861,0.694043,0.694043,0.695064,0.695064,0.731342,1.349043,0.798627,0.746766,0.749431,0.767623,0.714717,1.305651,0.843757,0.774391,0.754670,0.793205,0.700117,1.441584,1.046897,0.919697,0.847329,0.828725,0.685784,1.519906,1.237255,1.142440,0.922647,0.891547,0.671734,1.593176,1.516936,1.542689,0.971024,0.938128,0.657412,1.540569,1.489950,1.590915,1.240147,0.956071,0.643089,1.352698,1.391868,1.502407,1.104733,0.967542,0.629370,1.219034,1.247759,1.455751,1.191510,0.975015,2711,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $458.6m of $2109.7m (21.7%)
Volume: 644.5 of 1948.9 million kilograms (33.1%)
Value-to-Weight: 0.71 from 1.08 US$ per kg (+-34.3%)
Average distance to market (volume-weighted): 6941.7 from 5708.1 nm (+21.6%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    418  201399294.9  324645108.0      9.5     16.7
1               Fruits     62  155531529.6  102073949.3      7.4      5.2
2      Sugar and Honey     78   42307595.0   87868965.5      2.0      4.5
3  Unprocessed Cereals     24   30195552.1  109635464.0      1.4      5.6
4        Meat and Prep     78   15347353.6    8414320.0      0.7      0.4
5                Dairy    132    6786339.1    3137332.6      0.3      0.2
6           Vegetables    189    4565986.8    6848561.6      0.2      0.4
7        Fats and Oils     90    2440919.7    1841043.2      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,5786628.9,10658562.7,6316605.8,5907073.4,5928203.4,6072837.6,5316380.9,1.258770e+07,1.198462e+07,1.219046e+07,7680673.2,7417693.3,4982215.8,9637473.1,9866140.5,1.150548e+07,9419960.1,7708550.0
1,Fats and Oils,2634646.5,4615223.7,2844986.7,2665649.1,2674410.2,2752305.7,2442510.3,5.355394e+06,5.122812e+06,5.249778e+06,3439871.9,3301835.5,2306661.3,4225048.5,4331430.2,5.038895e+06,4136931.8,3419681.7
2,Fruits,306562439.1,565486692.9,334766515.4,313027715.6,314144792.0,321770708.8,281576456.3,6.678216e+08,6.358636e+08,6.466586e+08,407031682.6,393241717.3,263818309.5,510991084.7,523032361.5,6.102168e+08,499453478.0,408703741.8
3,Meat and Prep,12642067.0,23319716.8,13805165.3,12908692.2,12954758.1,13269233.1,11611682.9,2.753984e+07,2.622195e+07,2.666710e+07,16785247.1,16216590.7,10879363.5,21072366.9,21568922.2,2.516429e+07,20596585.8,16854221.8
4,Processed Cereals,569391013.3,937820118.8,604034050.5,569820657.2,572376816.9,591264107.0,533140763.9,1.107733e+09,1.056829e+09,1.089096e+09,744486820.1,692299510.7,507690031.7,898392015.5,929657813.9,1.044691e+09,876653901.6,712074708.5
5,Sugar and Honey,85760310.1,145179606.9,91599718.1,86227396.8,86594703.6,89280063.8,79942771.0,1.716039e+08,1.635585e+08,1.679783e+08,112516407.0,105620202.4,75844345.2,137041123.1,141488348.2,1.603813e+08,133836733.6,108917154.4
6,Unprocessed Cereals,166847918.8,307678322.9,182183874.4,170357380.3,170965741.7,175119970.4,153257338.7,3.633584e+08,3.459719e+08,3.518569e+08,221519511.6,213992929.3,143598433.5,278068359.1,284629034.8,3.320420e+08,271788815.0,222401023.8
7,Vegetables,9301688.2,17020255.3,10136224.5,9484052.4,9518262.4,9755289.5,8556445.5,2.009534e+07,1.913021e+07,1.947682e+07,12329129.8,11886140.3,8026564.8,15435028.9,15807675.9,1.840580e+07,15083618.4,12346208.9


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.72,0.08,0.02,0.02,0.04,1.07,0.98,1.01,0.35,0.31,0.69,0.72,0.96,0.65,0.40
1,Fats and Oils,0.81,0.09,0.01,0.02,0.05,1.19,1.10,1.15,0.41,0.35,0.79,0.83,1.12,0.75,0.46
2,Fruits,1.66,0.18,0.04,0.05,0.10,2.48,2.28,2.35,0.81,0.72,1.59,1.67,2.23,1.52,0.93
3,Meat and Prep,0.70,0.08,0.02,0.02,0.04,1.04,0.95,0.98,0.34,0.30,0.66,0.70,0.93,0.63,0.39
4,Processed Cereals,1.83,0.17,0.00,0.01,0.11,2.85,2.60,2.76,1.05,0.79,1.94,2.10,2.67,1.83,1.01
5,Sugar and Honey,1.40,0.14,0.01,0.02,0.08,2.17,1.98,2.08,0.77,0.61,1.45,1.55,2.00,1.37,0.78
6,Unprocessed Cereals,4.66,0.51,0.12,0.14,0.27,6.96,6.38,6.58,2.26,2.01,4.45,4.67,6.24,4.25,2.61
7,Vegetables,1.69,0.18,0.04,0.05,0.10,2.53,2.32,2.39,0.83,0.73,1.62,1.70,2.27,1.55,0.95
